# BIDSReader Tutorial

This tutorial covers all the public features of the **bidsreader** package — a Python library for reading and working with neuroimaging data stored in the [BIDS (Brain Imaging Data Structure)](https://bids.neuroimaging.io/) format.

### What you'll learn
1. **Setup** — Initializing a reader and configuring fields
2. **Dataset & Subject Metadata** — Querying subjects, tasks, and sessions
3. **Loading Events** — Behavioral and device-type event files
4. **Loading Electrodes & Channels** — Electrode coordinates, channel metadata, and combined views
5. **Loading Coordinate System Descriptions** — Spatial reference metadata
6. **Loading Raw EEG Data** — Continuous recordings via MNE
7. **Loading Epochs** — Time-locked segments around events
8. **Filtering by Trial Type** — Selecting specific event types across data objects
9. **Unit Detection & Conversion** — Detecting and converting EEG signal units
10. **MNE to PTSA Conversion** — Converting between MNE and PTSA TimeSeries formats
11. **Error Handling** — Working with the custom exception hierarchy

### Prerequisites
- Python 3.10+
- `mne`, `mne-bids`, `pandas`, `numpy` installed
- Access to a BIDS-formatted dataset (examples use the CML dataset at `/data/LTP_BIDS/`)
- Optional: `ptsa` for TimeSeries conversion examples

---
## 1. Setup & Imports

The bidsreader package exports all public classes and functions from its top-level module.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import mne

# Core reader classes
from bidsreader import BaseReader, CMLBIDSReader

# Filtering functions
from bidsreader import (
    filter_events_df_by_trial_types,
    filter_raw_events_by_trial_types,
    filter_epochs_by_trial_types,
    filter_by_trial_types,
)

# MNE <-> PTSA conversion
from bidsreader import mne_epochs_to_ptsa, mne_raw_to_ptsa

# Unit utilities
from bidsreader import detect_unit, get_scale_factor, convert_unit

# Exception hierarchy
from bidsreader.exc import (
    BIDSReaderError,
    InvalidOptionError,
    MissingRequiredFieldError,
    FileNotFoundBIDSError,
    AmbiguousMatchError,
    DataParseError,
    ExternalLibraryError,
)

### Initializing a CMLBIDSReader

`CMLBIDSReader` is the concrete reader for the CML (Computational Memory Lab) dataset. It extends `BaseReader` with CML-specific auto-detection and loading methods.

**Constructor fields:**
| Field | Type | Description |
|-------|------|-------------|
| `root` | str or Path | Path to the BIDS study root (defaults to `/data/LTP_BIDS/`) |
| `subject` | str | Subject ID (e.g., `"LTP606"`, `"R1111M"`) |
| `task` | str | Experiment/task name (e.g., `"FR1"`, `"valuecourier"`) |
| `session` | str or int | Session identifier |
| `device` | str | `"eeg"` (scalp) or `"ieeg"` (intracranial) — auto-detected if not set |
| `space` | str | Coordinate system (e.g., `"MNI152NLin6ASym"`) — auto-detected if not set |
| `acquisition` | str | `"monopolar"` or `"bipolar"` (iEEG only) |

In [ ]:
# Initialize a reader for scalp EEG data
# device is auto-detected from subject prefix: "LTP" -> eeg, "R" -> ieeg
reader = CMLBIDSReader(subject="LTP606", task="valuecourier", session=0)
print(reader)
print(f"Device (auto-detected): {reader.device}")

In [ ]:
# Initialize a reader for intracranial EEG (iEEG) data
ieeg_reader = CMLBIDSReader(root="/data/LTP_BIDS/FR1", subject="R1111M", task="FR1", session=0)
print(ieeg_reader)
print(f"Device (auto-detected): {ieeg_reader.device}")
print(f"Space (auto-detected): {ieeg_reader.space}")
print(f"Is intracranial: {ieeg_reader.is_intracranial()}")

### Changing fields after creation with `set_fields()`

Use `set_fields()` to update multiple reader fields at once. The method is chainable.

In [ ]:
# Update session and task
reader.set_fields(session=1, task="FR2")
print(reader)

# Switch back for the rest of the tutorial
reader.set_fields(session=0, task="FR1")

### String representations

Readers have both human-readable (`str`) and debug (`repr`) representations.

In [ ]:
print("str: ", str(reader))
print("repr:", repr(reader))

---
## 2. Dataset & Subject Metadata Queries

`BaseReader` (and therefore `CMLBIDSReader`) provides methods for discovering what's in your dataset without loading any data files.

In [ ]:
# List all subjects in the dataset
subjects = reader.get_dataset_subjects()
print(f"Subjects ({len(subjects)}): {subjects[:10]}...")

In [ ]:
# List all tasks in the dataset
tasks = reader.get_dataset_tasks()
print(f"Tasks: {tasks}")

In [ ]:
# List sessions and tasks for the current subject
sessions = reader.get_subject_sessions()
subject_tasks = reader.get_subject_tasks()
print(f"Subject {reader.subject} sessions: {sessions}")
print(f"Subject {reader.subject} tasks: {subject_tasks}")

In [ ]:
# Get the highest session number across all subjects
# Use outlier_thresh to exclude anomalously high session numbers
max_ses = reader.get_dataset_max_sessions(outlier_thresh=100)
print(f"Max session number: {max_ses}")

---
## 3. Loading Events

BIDS datasets store events in two locations:
- **`beh/`** folder: Behavioral events with `mstime` (millisecond timestamps)
- **`eeg/` or `ieeg/`** folder: Device events with `onset` (seconds), `duration`, and `sample` (digital sample index)

Use `load_events()` with the `event_type` parameter to choose which one to load.

In [ ]:
# Load behavioral events (default)
evs_beh = reader.load_events(event_type="beh")
print(f"Behavioral events shape: {evs_beh.shape}")
evs_beh.head()

In [ ]:
# Load device-type events (eeg or ieeg, depending on device)
evs_eeg = reader.load_events(event_type="eeg")
print(f"EEG events shape: {evs_eeg.shape}")
print(f"\nExtra columns in EEG events: {set(evs_eeg.columns) - set(evs_beh.columns)}")
evs_eeg.head()

The key difference is that the **eeg/ieeg** version includes `onset` (seconds), `duration`, and `sample` columns needed for MNE integration, while the **beh** version uses `mstime` (milliseconds).

### Common event types (trial_type column)
- `WORD` — word presentation during encoding
- `REC_WORD` — correctly recalled word during retrieval
- `REC_START` / `REC_STOP` — recall period boundaries
- `SESS_START` / `SESS_END` — session boundaries

In [ ]:
# Inspect available trial types
print("Trial types:")
print(evs_beh['trial_type'].value_counts())

---
## 4. Loading Electrodes & Channels (iEEG)

Intracranial EEG datasets contain rich metadata about electrode placement and channel configuration. These examples use the iEEG reader.

### Electrodes
The electrodes file contains spatial coordinates and region labels for each contact.

In [ ]:
elec_df = ieeg_reader.load_electrodes()
print(f"Electrodes shape: {elec_df.shape}")
elec_df.head()

### Channels

For iEEG data, channels come in two acquisition types:
- **monopolar**: Individual contacts measuring voltage relative to a common reference
- **bipolar**: Voltage differences between neighboring contact pairs

In [ ]:
# Monopolar channels
mono_df = ieeg_reader.load_channels(acquisition="monopolar")
print(f"Monopolar channels: {mono_df.shape}")
mono_df.head()

In [ ]:
# Bipolar channels (pairs)
bi_df = ieeg_reader.load_channels(acquisition="bipolar")
print(f"Bipolar channels: {bi_df.shape}")
bi_df.head()

### Combined channels

`load_combined_channels()` merges channel metadata with electrode coordinate data into a single DataFrame.

For **monopolar** data, this is a simple left-join on `name`. For **bipolar** data, it splits the pair name (e.g., `"LA1-LA2"`), looks up each contact's coordinates, computes midpoints, and checks region agreement.

In [ ]:
# Combined monopolar: channels + electrode coordinates
combined_mono = ieeg_reader.load_combined_channels(acquisition="monopolar")
print(f"Combined monopolar shape: {combined_mono.shape}")
combined_mono.head()

In [ ]:
# Combined bipolar: pairs + midpoint coordinates + region agreement
combined_bi = ieeg_reader.load_combined_channels(acquisition="bipolar")
print(f"Combined bipolar shape: {combined_bi.shape}")
# Note the x_mid, y_mid, z_mid columns for bipolar midpoints
midpoint_cols = [c for c in combined_bi.columns if '_mid' in c]
print(f"Midpoint columns: {midpoint_cols}")
combined_bi.head()

---
## 5. Loading Coordinate System Descriptions

The coordinate system JSON file describes the spatial reference frame used for electrode coordinates.

In [ ]:
coordsystem = ieeg_reader.load_coordsystem_desc()
print(f"Coordinate system keys: {list(coordsystem.keys())}")
coordsystem

---
## 6. Loading Raw EEG Data

`load_raw()` returns an `mne.io.BaseRaw` object containing the full continuous recording. You'll rarely need raw data directly — `load_epochs()` is more common — but it's useful for custom preprocessing or inspection.

For iEEG, you must specify `acquisition` (`"monopolar"` or `"bipolar"`).

In [ ]:
raw = ieeg_reader.load_raw(acquisition="monopolar")
print(f"Type: {type(raw).__name__}")
print(f"Channels: {len(raw.ch_names)}")
print(f"Duration: {raw.times[-1]:.1f} seconds")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
raw

---
## 7. Loading Epochs

`load_epochs()` creates time-locked EEG segments around events. This is the primary method for most EEG analyses.

**Parameters:**
| Parameter | Type | Description |
|-----------|------|-------------|
| `tmin` | float | Start of epoch window in seconds (relative to event) |
| `tmax` | float | End of epoch window in seconds |
| `events` | DataFrame, optional | Custom events DataFrame (must have `sample` column). If None, uses all annotations from the raw file |
| `baseline` | tuple, optional | Baseline correction window, e.g. `(None, 0)` |
| `acquisition` | str, optional | `"monopolar"` or `"bipolar"` (iEEG only) |
| `event_repeated` | str | How to handle duplicate events (default: `"merge"`) |
| `channels` | list, optional | Subset of channels to load |
| `preload` | bool | Whether to load data into memory immediately |

In [ ]:
# Load epochs: 0 to 1.6 seconds after each event
epochs = ieeg_reader.load_epochs(
    tmin=0, 
    tmax=1.6, 
    acquisition="monopolar", 
    preload=True,
)
print(f"Epochs shape: {epochs.get_data().shape}  (n_epochs, n_channels, n_times)")
print(f"Event IDs: {list(epochs.event_id.keys())[:10]}...")
epochs

In [ ]:
# Load epochs with a custom events DataFrame
# Only include events that have a 'sample' column
evs_ieeg = ieeg_reader.load_events(event_type="ieeg")
word_evs = evs_ieeg[evs_ieeg['trial_type'] == 'WORD'].copy()

word_epochs = ieeg_reader.load_epochs(
    tmin=-0.5,
    tmax=1.6,
    events=word_evs,
    acquisition="monopolar",
    preload=True,
)
print(f"Word epochs: {word_epochs.get_data().shape}")

In [ ]:
# Load epochs with channel selection and baseline correction
first_5_channels = mono_df['name'].tolist()[:5]

epochs_subset = ieeg_reader.load_epochs(
    tmin=-0.25,
    tmax=1.6,
    acquisition="monopolar",
    baseline=(None, 0),  # baseline correct using pre-event period
    channels=first_5_channels,
    preload=True,
)
print(f"Subset epochs shape: {epochs_subset.get_data().shape}")
print(f"Channels: {epochs_subset.ch_names}")

---
## 8. Filtering by Trial Type

BIDSReader provides four filtering functions for selecting specific event types. These handle MNE's merged event labels (e.g., `"WORD/STIM"`) automatically by matching individual tokens.

### 8a. Filter an events DataFrame

In [ ]:
# filter_events_df_by_trial_types returns (filtered_df, indices_into_original)
filtered_df, df_indices = filter_events_df_by_trial_types(evs_ieeg, ["WORD"])
print(f"Original events: {len(evs_ieeg)}")
print(f"WORD events: {len(filtered_df)}")
print(f"Index positions: {df_indices[:5]}...")
filtered_df.head()

### 8b. Filter MNE Raw annotations

In [ ]:
# filter_raw_events_by_trial_types returns (events_array, event_id_dict, indices)
filtered_events, filtered_event_id, raw_indices = filter_raw_events_by_trial_types(raw, ["WORD"])
print(f"Filtered raw events shape: {filtered_events.shape}")
print(f"Event IDs: {filtered_event_id}")

### 8c. Filter MNE Epochs

In [ ]:
# filter_epochs_by_trial_types returns (filtered_epochs, event_id_dict, indices)
filtered_epochs, ep_event_id, ep_indices = filter_epochs_by_trial_types(epochs, ["WORD"])
print(f"Original epochs: {len(epochs)}")
print(f"WORD epochs: {len(filtered_epochs)}")
print(f"Event IDs: {ep_event_id}")

### 8d. Filter everything at once with consistency checks

`filter_by_trial_types()` filters across multiple data objects simultaneously and verifies that trial counts and event onsets match. This catches data alignment issues early.

In [ ]:
# filter_by_trial_types returns a 5-tuple:
# (filtered_df, filtered_raw_events, filtered_epochs, event_id, event_indices)
filt_df, filt_raw_events, filt_epochs, filt_event_id, filt_idx = filter_by_trial_types(
    ["WORD"],
    events_df=evs_ieeg,
    raw=raw,
    epochs=epochs,
)

print(f"Filtered DataFrame: {len(filt_df)} rows")
print(f"Filtered raw events: {filt_raw_events.shape}")
print(f"Filtered epochs: {len(filt_epochs)}")
print(f"Consistent event_id: {filt_event_id}")
print(f"Event indices (0..n-1): {filt_idx[:5]}...")

In [ ]:
# You can also pass only some of the data objects (the rest default to None)
filt_df_only, _, _, _, _ = filter_by_trial_types(
    ["WORD", "REC_WORD"],
    events_df=evs_ieeg,
)
print(f"WORD + REC_WORD events: {len(filt_df_only)}")

---
## 9. Unit Detection & Conversion

EEG data can be stored in different voltage units (V, mV, uV, nV, etc.). The `units` module provides tools to detect, convert, and compute scale factors between units.

### Detecting units

In [ ]:
# Detect the unit stored in an MNE object's FIFF metadata
current_unit = detect_unit(raw)
print(f"Raw data unit: {current_unit}")

# You can also override auto-detection with a known unit
explicit_unit = detect_unit(raw, current_unit="uV")
print(f"Explicit unit: {explicit_unit}")

### Computing scale factors

In [ ]:
# Get the multiplicative factor to convert between units
factor_v_to_uv = get_scale_factor("V", "uV")
print(f"V -> uV: multiply by {factor_v_to_uv}")

factor_uv_to_mv = get_scale_factor("uV", "mV")
print(f"uV -> mV: multiply by {factor_uv_to_mv}")

factor_nv_to_v = get_scale_factor("nV", "V")
print(f"nV -> V: multiply by {factor_nv_to_v}")

### Converting data units

`convert_unit()` scales the data and updates the unit metadata on MNE objects (FIFF channel info) or PTSA TimeSeries (attrs).

In [ ]:
# Convert raw data to microvolts (returns a copy by default)
raw_uv = convert_unit(raw, "uV")
print(f"Original unit: {detect_unit(raw)}")
print(f"Converted unit: {detect_unit(raw_uv)}")

In [ ]:
# Convert epochs too
epochs_uv = convert_unit(epochs, "uV")
print(f"Epochs unit: {detect_unit(epochs_uv)}")

In [ ]:
# Use copy=False to modify in-place (saves memory for large datasets)
# raw_inplace = convert_unit(raw.copy(), "mV", copy=False)

### Supported units

Both **Volts** and **Tesla** are supported across the full SI prefix range:

| Prefix | Symbol | Factor |
|--------|--------|--------|
| peta   | PV     | 10^15  |
| giga   | GV     | 10^9   |
| mega   | MV     | 10^6   |
| kilo   | kV     | 10^3   |
| (base) | V      | 10^0   |
| milli  | mV     | 10^-3  |
| micro  | uV     | 10^-6  |
| nano   | nV     | 10^-9  |
| pico   | pV     | 10^-12 |
| femto  | fV     | 10^-15 |

Unicode characters (µ, μ, Ω, °) are automatically normalized.

---
## 10. MNE to PTSA Conversion

If you use the [PTSA (Penn Time Series Analysis)](https://github.com/pennmem/ptsa_new) library, bidsreader provides conversion functions.

### Converting Raw data

In [ ]:
# Convert full raw recording to PTSA TimeSeries
raw_ts = mne_raw_to_ptsa(raw)
print(f"Type: {type(raw_ts).__name__}")
print(f"Dims: {raw_ts.dims}")
print(f"Shape: {raw_ts.shape}")
raw_ts

In [ ]:
# Convert with channel selection and time cropping
raw_ts_subset = mne_raw_to_ptsa(
    raw, 
    picks=[raw.ch_names[0], raw.ch_names[1]],  # first 2 channels
    tmin=0.0, 
    tmax=10.0,  # first 10 seconds
)
print(f"Subset shape: {raw_ts_subset.shape}")
print(f"Channels: {list(raw_ts_subset.coords['channel'].values)}")

### Converting Epochs

In [ ]:
# Convert MNE Epochs to PTSA TimeSeries
# Requires the events DataFrame (with 'sample' column) for alignment
evs_ieeg = ieeg_reader.load_events(event_type="ieeg")
epochs_ts = mne_epochs_to_ptsa(epochs, evs_ieeg)
print(f"Type: {type(epochs_ts).__name__}")
print(f"Shape: {epochs_ts.shape}")
epochs_ts

---
## 11. Error Handling

All bidsreader exceptions inherit from `BIDSReaderError`, so you can catch them all with a single handler or handle specific cases.

The `@public_api` decorator on all public methods automatically maps common external exceptions to the bidsreader hierarchy:
- `FileNotFoundError` → `FileNotFoundBIDSError`
- `json.JSONDecodeError` / `pd.errors.ParserError` → `DataParseError`
- `KeyError` → `DataParseError`
- Other exceptions → `ExternalLibraryError`

```
BIDSReaderError
├── InvalidOptionError          # bad argument value
├── MissingRequiredFieldError    # required field not set
├── FileNotFoundBIDSError        # expected BIDS file missing
├── AmbiguousMatchError          # multiple files matched
├── DataParseError               # TSV/JSON parse failure
├── DependencyError              # optional dep issue
└── ExternalLibraryError         # unexpected error from MNE/pandas
```

In [ ]:
# MissingRequiredFieldError: loading without required fields
try:
    bad_reader = CMLBIDSReader(subject="LTP606", task="valuecourier")
    bad_reader.session = None
    bad_reader.load_events()
except MissingRequiredFieldError as e:
    print(f"MissingRequiredFieldError: {e}")

In [ ]:
# InvalidOptionError: passing an invalid device type
try:
    CMLBIDSReader(subject="LTP606", task="test", session=0, device="meg")
except InvalidOptionError as e:
    print(f"InvalidOptionError: {e}")

In [ ]:
# FileNotFoundBIDSError: trying to load from a non-existent path
try:
    missing_reader = CMLBIDSReader(
        root="/tmp/nonexistent_bids", 
        subject="X999", 
        task="test", 
        session=0,
        device="eeg",
    )
    missing_reader.load_events()
except FileNotFoundBIDSError as e:
    print(f"FileNotFoundBIDSError: {e}")

In [ ]:
# Catch all bidsreader exceptions with a single handler
try:
    CMLBIDSReader(subject="LTP606", task="test", session=0, device="meg")
except BIDSReaderError as e:
    print(f"Caught {type(e).__name__}: {e}")

---
## Summary

Here's a quick reference of everything covered in this tutorial:

### Reader Classes
| Class | Description |
|-------|-------------|
| `BaseReader` | Abstract base class — metadata queries, field validation, BIDS path construction |
| `CMLBIDSReader` | Concrete reader for CML datasets — auto-detects device/space, loads all BIDS file types |

### CMLBIDSReader Methods
| Method | Returns |
|--------|--------|
| `set_fields(**kwargs)` | self (chainable) |
| `is_intracranial()` | bool |
| `get_dataset_subjects()` | list[str] |
| `get_dataset_tasks()` | list[str] |
| `get_subject_sessions()` | list[str] |
| `get_subject_tasks()` | list[str] |
| `get_dataset_max_sessions()` | int or None |
| `load_events(event_type)` | DataFrame |
| `load_electrodes()` | DataFrame |
| `load_channels(acquisition)` | DataFrame |
| `load_combined_channels(acquisition)` | DataFrame |
| `load_coordsystem_desc()` | dict |
| `load_raw(acquisition)` | mne.io.BaseRaw |
| `load_epochs(tmin, tmax, ...)` | mne.Epochs |

### Standalone Functions
| Function | Module | Purpose |
|----------|--------|---------|
| `filter_events_df_by_trial_types()` | filtering | Filter DataFrame by trial type |
| `filter_raw_events_by_trial_types()` | filtering | Filter MNE Raw annotations |
| `filter_epochs_by_trial_types()` | filtering | Filter MNE Epochs |
| `filter_by_trial_types()` | filtering | Filter all + consistency checks |
| `detect_unit()` | units | Detect/validate EEG unit |
| `get_scale_factor()` | units | Compute conversion factor |
| `convert_unit()` | units | Convert data to target unit |
| `mne_raw_to_ptsa()` | convert | Raw -> PTSA TimeSeries |
| `mne_epochs_to_ptsa()` | convert | Epochs -> PTSA TimeSeries |

### Exceptions
All inherit from `BIDSReaderError` — catch the base class for blanket handling, or specific subclasses for targeted error recovery.